# IonQ native cloud: linear XEB from existing OpenQASM 3 files

Standalone **XEB-only** runner. It does **not** submit VQC or QAOA circuits.

| Item | Detail |
|------|--------|
| Circuits | `xeb_dDD_cCCC.qasm` under `QASM_DIR` (default: `data/circuits/`) |
| Metric | Linear cross-entropy benchmarking fidelity \(F_{\mathrm{XEB}}\) vs ideal Born probabilities |
| Provider | [`qiskit-ionq`](https://docs.ionq.com/sdks/qiskit) → IonQ Quantum Cloud |
| Outputs | `data/raw/ionq_native/<RUN_STAMP>/<device_slug>/xeb/` |

**Related notebooks**

- VQC + QAOA (no XEB): [`ionq-native-qaoa-and-vqc-from-qasm.ipynb`](ionq-native-qaoa-and-vqc-from-qasm.ipynb)
- VQC + XEB combined (legacy): [`ionq-benchmarks-from-qasm.ipynb`](ionq-benchmarks-from-qasm.ipynb)
- Ideal reference (local, no cloud): `analysis/ideal_reference_xeb_from_qasm.ipynb`

**Credentials:** set `export IONQ_API_KEY=...` or paste into the config cell.

**Recommended order:** `simulator` (free) → optional `simulator` + `noise_model="forte-1"` → `qpu.forte-1` (credits + queue).


In [ ]:
# Run once per fresh kernel.
%pip install -q "qiskit>=1.2,<2" qiskit-qasm3-import "qiskit-ionq<1" scipy matplotlib


In [ ]:
from __future__ import annotations

import json
import os
import re
import time
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import curve_fit

from qiskit import QuantumCircuit, transpile, qasm3
from qiskit.quantum_info import Statevector
from qiskit.transpiler.passes import RemoveBarriers
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager


## Configuration — edit this cell only


In [ ]:
# ---------------------------------------------------------------------------
# Timestamp folder.  None -> UTC now.  Set explicitly to attach XEB results to
# an existing data/raw/ionq_native/<RUN_STAMP>/ tree (e.g. "20260707_084925Z").
# ---------------------------------------------------------------------------
RUN_STAMP: Optional[str] = None
if RUN_STAMP is None:
    RUN_STAMP = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%SZ")

OUTPUT_ROOT = Path("../data/raw/ionq_native") / RUN_STAMP
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print("Results will be written under:", OUTPUT_ROOT.resolve())

# ---------------------------------------------------------------------------
# IonQ backends to benchmark (XEB only).
# ---------------------------------------------------------------------------
IONQ_DEVICES: List[Dict[str, Any]] = [
    {"backend": "simulator", "slug": "simulator_ideal"},
    # {"backend": "simulator", "noise_model": "forte-1", "slug": "simulator_forte_1"},
    # {"backend": "qpu.forte-1", "slug": "qpu_forte_1"},
]

IONQ_API_KEY: Optional[str] = None

# Shots per XEB circuit.
SHOTS_XEB = 4000

# Limit circuits per depth (None = all xeb_d*_c*.qasm at that depth).
XEB_PER_DEPTH_LIMIT: Optional[int] = None

# Restrict to these cycle depths (None = all depths found in filenames).
XEB_DEPTHS: Optional[List[int]] = None  # e.g. [1] or [1, 3, 4]

# Circuit index cNNN in filename (inclusive). None = no bound.
# Resume example after c000–c004 finished: MIN=5, MAX=10
XEB_CIRCUIT_IDX_MIN: Optional[int] = None
XEB_CIRCUIT_IDX_MAX: Optional[int] = None

# Skip submission when counts/<stem>.json already exists in the output folder.
XEB_SKIP_IF_COUNTS_EXIST = True

# Circuits per IonQ job. QPU backends force 1 circuit/job internally (API quirk).
IONQ_BATCH_SIZE: Optional[int] = 25

# OpenQASM source folder.  Default auto-finds data/circuits/.
# For IBM Marrakesh depth sweep: Path("ibm_xeb_qasm_jobs_marrakesh")
QASM_DIR: Optional[Path] = None

TRANSPILE_OPTLEVEL = 1
NUM_QUBITS = 10
IONQ_JOB_TIMEOUT_S: Optional[float] = None
IONQ_POLL_S = 5.0


## IonQ provider (lists backends if login works)


In [ ]:
def _resolve_ionq_api_key() -> str:
    explicit = globals().get("IONQ_API_KEY")
    if explicit:
        return str(explicit)
    for var in ("IONQ_API_KEY", "IONQ_API_TOKEN"):
        v = os.environ.get(var)
        if v:
            return v
    raise RuntimeError(
        "No IonQ API key. Set IONQ_API_KEY in the config cell or export it in the shell."
    )


from qiskit_ionq import IonQProvider  # noqa: E402

_PROVIDER = IonQProvider(_resolve_ionq_api_key())
print("IonQ provider initialised.")

try:
    names = sorted({b.name for b in _PROVIDER.backends()})
    print(f"{len(names)} visible backend(s):")
    for n in names:
        print(" ", repr(n))
except Exception as exc:
    print("Could not list backends:", exc)


## Helpers — QASM load + linear XEB fidelity


In [ ]:
_XEB_FNAME_RE = re.compile(r"xeb_d(\d+)_c(\d+)\.qasm$")
_REMOVE_BARRIERS = RemoveBarriers()


def _resolve_qasm_dir() -> Path:
    explicit = globals().get("QASM_DIR")
    if explicit is not None:
        p = Path(explicit).expanduser().resolve()
        if not p.is_dir():
            raise FileNotFoundError(f"QASM_DIR does not exist: {p}")
        return p
    here = Path.cwd().resolve()
    for c in [
        here.parent / "data" / "circuits",
        here.parent / "data" / "circuits",
        here / "ibm_xeb_qasm_jobs_marrakesh",
        here.parent / "data" / "circuits",
        here.parent / "data" / "circuits",
    ]:
        if c.is_dir():
            return c.resolve()
    raise FileNotFoundError(
        "Could not find XEB QASM folder. Set QASM_DIR explicitly."
    )


def _parse_xeb_depth_idx(path: Path) -> Tuple[int, int]:
    m = _XEB_FNAME_RE.search(path.name)
    if not m:
        raise ValueError(f"Not an XEB QASM file: {path.name}")
    return int(m.group(1)), int(m.group(2))


def _circuit_idx_in_range(cidx: int) -> bool:
    lo = globals().get("XEB_CIRCUIT_IDX_MIN")
    hi = globals().get("XEB_CIRCUIT_IDX_MAX")
    if lo is not None and cidx < int(lo):
        return False
    if hi is not None and cidx > int(hi):
        return False
    return True


def _paths_for_depth(qasm_dir: Path, depth: int) -> List[Path]:
    return sorted(
        qasm_dir.glob(f"xeb_d{depth:02d}_c*.qasm"),
        key=_parse_xeb_depth_idx,
    )


def load_qasm3_circuit(path: Path) -> QuantumCircuit:
    qc = qasm3.load(str(path))
    if qc.num_qubits != NUM_QUBITS:
        raise ValueError(f"{path.name}: expected {NUM_QUBITS} qubits, got {qc.num_qubits}")
    return _REMOVE_BARRIERS(qc)


def strip_final_measurements(qc: QuantumCircuit) -> QuantumCircuit:
    return qc.remove_final_measurements(inplace=False)


def ideal_probabilities(circuit_no_meas: QuantumCircuit) -> np.ndarray:
    return np.abs(Statevector.from_instruction(circuit_no_meas).data) ** 2


def _normalise_counts(counts: Dict[str, Any], shots: int) -> Dict[str, int]:
    if not counts:
        return {}
    total = sum(counts.values())
    if total <= 1.5:
        return {k.replace(" ", ""): int(round(v * shots)) for k, v in counts.items()}
    return {k.replace(" ", ""): int(v) for k, v in counts.items()}


def xeb_fidelity_from_counts(p_ideal: np.ndarray, counts: Dict[str, int]) -> float:
    D = len(p_ideal)
    total = float(sum(counts.values()))
    if total <= 0:
        return float("nan")
    p_meas = np.zeros(D, dtype=float)
    for b, n in counts.items():
        idx = int(b, 2)
        p_meas[idx] += n / total
    num = float(np.sum(p_meas * p_ideal) - 1.0 / D)
    den = float(np.sum(p_ideal ** 2) - 1.0 / D)
    return num / den


## IonQ submit + XEB runner


In [ ]:
def _device_slug(cfg: Dict[str, Any]) -> str:
    if cfg.get("slug"):
        return str(cfg["slug"])
    base = str(cfg["backend"])
    nm = cfg.get("noise_model")
    candidate = f"{base}__{nm}" if nm else base
    return re.sub(r"[^0-9A-Za-z._-]+", "_", candidate)[:120]


def _get_ionq_backend(cfg: Dict[str, Any]):
    name = cfg["backend"]
    backend = _PROVIDER.get_backend(name)
    nm = cfg.get("noise_model")
    if nm is not None:
        if not name.endswith("simulator"):
            raise ValueError(f"noise_model only valid on simulator, got {name!r}")
        backend.set_options(noise_model=nm)
    return backend


def _ionq_backend_name(backend) -> str:
    name = getattr(backend, "name", None)
    if callable(name):
        return str(name())
    if name is not None:
        return str(name)
    return str(backend)


def _transpile_for_ionq(backend, qc: QuantumCircuit) -> QuantumCircuit:
    target = getattr(backend, "target", None)
    if target is not None:
        pm = generate_preset_pass_manager(
            target=target, optimization_level=TRANSPILE_OPTLEVEL,
        )
        return pm.run(qc)
    return transpile(qc, backend=backend, optimization_level=TRANSPILE_OPTLEVEL)


def _looks_like_uuid(key: Any) -> bool:
    s = str(key)
    return "-" in s and len(s) >= 32


def _parse_outcome_key(key: Any) -> Optional[int]:
    s = str(key).strip()
    if _looks_like_uuid(s):
        return None
    if s.lower().startswith("0x"):
        try:
            return int(s, 16)
        except ValueError:
            return None
    try:
        return int(s)
    except ValueError:
        return None


def _coerce_weight(val: Any) -> Optional[float]:
    if isinstance(val, (int, float)):
        return float(val)
    if isinstance(val, str):
        try:
            return float(val)
        except ValueError:
            return None
    return None


def _extract_histogram(obj: Any) -> Optional[dict]:
    # Pull a numeric-key histogram out of assorted IonQ API shapes.
    if not isinstance(obj, dict) or not obj:
        return None
    for wrap in ("probabilities", "counts", "histogram", "data", "results"):
        if wrap in obj and isinstance(obj[wrap], dict):
            inner = _extract_histogram(obj[wrap])
            if inner:
                return inner
    pairs: List[Tuple[str, float]] = []
    for key, val in obj.items():
        if isinstance(val, dict):
            continue
        ok = _parse_outcome_key(key)
        w = _coerce_weight(val)
        if ok is not None and w is not None:
            pairs.append((str(ok), w))
    if pairs:
        return dict(pairs)
    best: Optional[dict] = None
    for val in obj.values():
        if isinstance(val, dict):
            inner = _extract_histogram(val)
            if inner and (best is None or len(inner) > len(best)):
                best = inner
    return best


def _split_ionq_histogram_payload(raw: Any) -> List[dict]:
    if isinstance(raw, list):
        out = [_extract_histogram(item) for item in raw]
        out = [h for h in out if h]
        if out:
            return out
    if isinstance(raw, dict) and raw:
        keys = list(raw.keys())
        if keys and all(_looks_like_uuid(k) for k in keys) and all(
            isinstance(v, dict) for v in raw.values()
        ):
            out = [_extract_histogram(v) for v in raw.values()]
            out = [h for h in out if h]
            if out:
                return out
            sample = next(iter(raw.values()))
            raise ValueError(
                "Could not parse per-circuit histograms under UUID keys. "
                f"Sample inner keys: {list(sample.keys())[:8]}"
            )
        single = _extract_histogram(raw)
        if single:
            return [single]
        if all(isinstance(v, dict) for v in raw.values()):
            out = [_extract_histogram(v) for v in raw.values()]
            out = [h for h in out if h]
            if out:
                return out
    raise ValueError(
        "Unrecognized IonQ results payload. "
        f"top-level keys sample: {list(raw.keys())[:3] if isinstance(raw, dict) else raw}"
    )


def _results_url_from_response(response: dict) -> Optional[str]:
    res = response.get("results")
    if isinstance(res, str):
        return res
    if isinstance(res, dict):
        prob = res.get("probabilities")
        if isinstance(prob, dict) and prob.get("url"):
            return str(prob["url"])
        if res.get("url"):
            return str(res["url"])
    ru = response.get("results_url")
    return str(ru) if isinstance(ru, str) else None


def _fetch_histograms_for_job(job) -> List[dict]:
    job.status()
    client = job._client
    children = getattr(job, "_children", None) or []
    if children:
        histograms: List[dict] = []
        for child_id in children:
            resp = client.retrieve_job(child_id)
            url = _results_url_from_response(resp)
            if not url:
                raise RuntimeError(
                    f"child job {child_id}: no results URL (keys={list(resp.keys())})"
                )
            histograms.extend(_split_ionq_histogram_payload(client.get_results(url)))
        return histograms
    url = getattr(job, "_results_url", None)
    if not url:
        raise RuntimeError(f"job {job.job_id()}: missing results URL after completion")
    return _split_ionq_histogram_payload(client.get_results(url))


def _ionq_hist_to_bitstring_counts(
    hist: dict,
    num_qubits: int,
    clbits: List[int],
    shots: int,
) -> Dict[str, int]:
    # Convert one IonQ histogram to Qiskit-style bitstring counts.

    def get_bitvalue(bitstring: str, bit: int) -> str:
        if bit is not None and 0 <= bit < len(bitstring):
            return bitstring[bit]
        return "0"

    mapped: Dict[int, float] = {}
    for value, weight in hist.items():
        ok = _parse_outcome_key(value)
        w = _coerce_weight(weight)
        if ok is None or w is None:
            continue
        bitstring = bin(ok)[2:].rjust(num_qubits, "0")[::-1]
        outvalue = int("".join(get_bitvalue(bitstring, bit) for bit in clbits)[::-1], 2)
        mapped[outvalue] = mapped.get(outvalue, 0.0) + w

    total = sum(mapped.values())
    is_prob = total <= 1.5
    counts: Dict[str, int] = {}
    for key, val in mapped.items():
        bits = bin(int(key))[2:].rjust(num_qubits, "0")
        count = int(round(val * shots)) if is_prob else int(round(val))
        if count:
            counts[bits] = count
    return counts


def _fetch_chunk_counts_ionq(job, chunk: List[QuantumCircuit], shots: int) -> List[Dict[str, int]]:
    histograms = _fetch_histograms_for_job(job)

    num_qubits = getattr(job, "_num_qubits", None) or NUM_QUBITS
    clbits_list = getattr(job, "_clbits", None) or [list(range(num_qubits))] * len(chunk)
    if len(clbits_list) == 1 and len(chunk) > 1:
        clbits_list = clbits_list * len(chunk)

    if len(histograms) != len(chunk):
        raise RuntimeError(
            f"job {job.job_id()}: expected {len(chunk)} histograms, got {len(histograms)}"
        )

    return [
        _ionq_hist_to_bitstring_counts(histograms[i], num_qubits, clbits_list[i], shots)
        for i in range(len(chunk))
    ]


def _effective_batch_size(backend, batch_size: Optional[int], n: int) -> int:
    name = _ionq_backend_name(backend).lower()
    if batch_size == 1:
        return 1
    if "qpu" in name or "forte" in name or "aria" in name:
        return 1
    return batch_size if (batch_size and batch_size > 0) else n


def _submit_chunks(
    backend,
    circuits: List[QuantumCircuit],
    shots: int,
    batch_size: Optional[int],
) -> Tuple[List[Dict[str, int]], List[str]]:
    if not circuits:
        return [], []
    n = len(circuits)
    bs = _effective_batch_size(backend, batch_size, n)
    if bs == 1 and n > 1 and "qpu" in _ionq_backend_name(backend).lower():
        print("    (QPU: submitting 1 circuit per IonQ job)")
    counts_all: List[Dict[str, int]] = []
    job_ids: List[str] = []
    for start in range(0, n, bs):
        chunk = circuits[start : start + bs]
        job = backend.run(chunk, shots=shots)
        jid = job.job_id()
        job_ids.append(jid)
        print(f"    submitted {len(chunk)} circuit(s) -> job_id={jid}")
        job.wait_for_final_state(timeout=IONQ_JOB_TIMEOUT_S, wait=IONQ_POLL_S)
        # qiskit-ionq 0.6.x mis-parses QPU multi-circuit jobs when histogram
        # weights are ints (UUID keys leak into map_output). Parse raw API JSON.
        chunk_counts = _fetch_chunk_counts_ionq(job, chunk, shots)
        if len(chunk_counts) != len(chunk):
            raise RuntimeError(
                f"job {jid}: expected {len(chunk)} count dicts, got {len(chunk_counts)}"
            )
        counts_all.extend(chunk_counts)
    return counts_all, job_ids


def discover_xeb_files(qasm_dir: Path) -> Dict[int, List[Path]]:
    files = sorted(qasm_dir.glob("xeb_d*_c*.qasm"), key=_parse_xeb_depth_idx)
    if not files:
        raise FileNotFoundError(f"No xeb_d*_c*.qasm under {qasm_dir}")
    groups: Dict[int, List[Path]] = {}
    for p in files:
        d, cidx = _parse_xeb_depth_idx(p)
        if not _circuit_idx_in_range(cidx):
            continue
        groups.setdefault(d, []).append(p)
    if not groups:
        raise ValueError(
            "No XEB circuits left after depth/index filters. "
            f"XEB_CIRCUIT_IDX_MIN={XEB_CIRCUIT_IDX_MIN}, XEB_CIRCUIT_IDX_MAX={XEB_CIRCUIT_IDX_MAX}"
        )
    if XEB_DEPTHS is not None:
        groups = {d: groups[d] for d in sorted(groups) if d in XEB_DEPTHS}
        if not groups:
            raise ValueError(f"No circuits for XEB_DEPTHS={XEB_DEPTHS}")
    if XEB_PER_DEPTH_LIMIT is not None:
        groups = {d: ps[:XEB_PER_DEPTH_LIMIT] for d, ps in groups.items()}
    return groups


def _summarize_depth_from_disk(
    depth: int, qasm_dir: Path, counts_dir: Path
) -> Optional[Dict[str, Any]]:
    fvals: List[float] = []
    files: List[str] = []
    for path in _paths_for_depth(qasm_dir, depth):
        cf = counts_dir / f"{path.stem}.json"
        if not cf.is_file():
            continue
        qc_meas = load_qasm3_circuit(path)
        p_ideal = ideal_probabilities(strip_final_measurements(qc_meas))
        cdict = json.loads(cf.read_text())
        fvals.append(xeb_fidelity_from_counts(p_ideal, cdict))
        files.append(path.name)
    if not fvals:
        return None
    return {
        "mean": float(np.mean(fvals)),
        "sem": float(np.std(fvals) / np.sqrt(len(fvals))) if len(fvals) > 1 else 0.0,
        "per_circuit": fvals,
        "files": files,
    }


def _all_xeb_depths(qasm_dir: Path) -> List[int]:
    depths = sorted(
        {_parse_xeb_depth_idx(p)[0] for p in qasm_dir.glob("xeb_d*_c*.qasm")}
    )
    if XEB_DEPTHS is not None:
        depths = [d for d in depths if d in XEB_DEPTHS]
    return depths


def rebuild_xeb_summary_from_disk(
    cfg: Dict[str, Any],
    out_xeb: Path,
    qasm_dir: Path,
    *,
    wall_time_s: float = 0.0,
) -> Dict[str, Any]:
    # Rebuild summary/plot from every counts/*.json on disk (merged partial runs).
    counts_dir = out_xeb / "counts"
    backend = _get_ionq_backend(cfg)
    per_depth: Dict[int, Dict[str, Any]] = {}
    for d in _all_xeb_depths(qasm_dir):
        agg = _summarize_depth_from_disk(d, qasm_dir, counts_dir)
        if agg is not None:
            per_depth[int(d)] = agg
            print(
                f"  [merge] depth {d}: {len(agg['files'])}/{len(_paths_for_depth(qasm_dir, d))} "
                f"circuits on disk, mean F={agg['mean']:.4f}"
            )
    if not per_depth:
        raise RuntimeError(
            f"No XEB count files under {counts_dir}. Run recovery or submit first."
        )

    job_ids_path = out_xeb / "job_ids.txt"
    all_job_ids: List[str] = []
    if job_ids_path.is_file():
        all_job_ids = [ln.strip() for ln in job_ids_path.read_text().splitlines() if ln.strip()]

    depths = sorted(per_depth.keys())
    depths_arr = np.array(depths, dtype=float)
    mean_f = np.array([per_depth[d]["mean"] for d in depths])
    sem_f = np.array([per_depth[d]["sem"] for d in depths])

    def model(dd, A, f):
        return A * f ** dd

    fit: Dict[str, Any] = {}
    if len(depths) >= 2:
        try:
            popt, pcov = curve_fit(
                model, depths_arr, mean_f, p0=[1.0, 0.99],
                sigma=np.maximum(sem_f, 1e-3), bounds=([0, 0], [1.5, 1.0]),
            )
            A_fit, f_fit = popt
            A_err, f_err = np.sqrt(np.diag(pcov))
            fit = {"A": A_fit, "A_err": A_err, "f_per_cycle": f_fit, "f_err": f_err}
        except Exception as e:
            fit = {"error": str(e)}
    else:
        fit = {"error": "need >=2 distinct depths to fit f^d"}

    summary = {
        "benchmark": "xeb_from_qasm",
        "provider": "ionq_native",
        "backend": _ionq_backend_name(backend),
        "noise_model": cfg.get("noise_model"),
        "backend_name_arg": cfg["backend"],
        "qasm_dir": str(qasm_dir),
        "shots": SHOTS_XEB,
        "depths": depths,
        "circuits_per_depth": {
            str(d): len(_paths_for_depth(qasm_dir, d)) for d in depths
        },
        "circuits_with_counts": {
            str(d): len(per_depth[d]["files"]) for d in per_depth
        },
        "per_depth_mean_sem": {
            str(k): {"mean": per_depth[k]["mean"], "sem": per_depth[k]["sem"]}
            for k in per_depth
        },
        "fit_A_f": fit,
        "wall_time_s": wall_time_s,
        "ionq_job_ids": all_job_ids,
        "utc_finished": datetime.now(timezone.utc).isoformat(),
    }
    (out_xeb / "summary.json").write_text(json.dumps(summary, indent=2))
    (out_xeb / "per_depth_mean_sem.json").write_text(
        json.dumps(summary["per_depth_mean_sem"], indent=2)
    )
    (out_xeb / "per_depth_per_circuit.json").write_text(
        json.dumps(
            {str(k): {"fidelities": v["per_circuit"], "files": v["files"]} for k, v in per_depth.items()},
            indent=2,
        )
    )

    plt.figure(figsize=(7, 4))
    plt.errorbar(depths, mean_f, yerr=sem_f, fmt="s-", label=r"$F_{\mathrm{XEB}}$")
    if "f_per_cycle" in fit:
        dd = np.linspace(min(depths), max(depths), 100)
        plt.plot(dd, model(dd, fit["A"], fit["f_per_cycle"]), "k:",
                 label=f"fit f={fit['f_per_cycle']:.4f}")
    plt.xlabel("cycles d")
    plt.ylabel(r"$F_{\mathrm{XEB}}$")
    tag = _ionq_backend_name(backend)
    if cfg.get("noise_model"):
        tag += f" / noise={cfg['noise_model']}"
    plt.title(f"Linear XEB — {tag}")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(out_xeb / "xeb_plot.png", dpi=150)
    plt.show()

    print(f"  [merge] wrote summary -> {out_xeb.resolve()}")
    return summary


def run_xeb_ionq(cfg: Dict[str, Any], out_xeb: Path, qasm_dir: Path) -> Dict[str, Any]:
    out_xeb.mkdir(parents=True, exist_ok=True)
    counts_dir = out_xeb / "counts"
    counts_dir.mkdir(exist_ok=True)

    backend = _get_ionq_backend(cfg)
    groups = discover_xeb_files(qasm_dir)
    depths = sorted(groups.keys())
    cpd = {d: len(ps) for d, ps in groups.items()}
    print(f"  [XEB] depths={depths}  circuits/depth={cpd}")

    t0 = time.time()
    new_job_ids: List[str] = []

    for d in depths:
        pending: List[Path] = []
        for path in groups[d]:
            stem = path.stem
            if XEB_SKIP_IF_COUNTS_EXIST and (counts_dir / f"{stem}.json").is_file():
                print(f"  [skip] {path.name} (counts already on disk)")
                continue
            pending.append(path)
        if not pending:
            print(f"  [XEB] depth {d}: no pending circuits to submit")
            continue

        circuits: List[QuantumCircuit] = []
        p_ideals: List[np.ndarray] = []
        circ_files: List[str] = []
        for path in pending:
            qc_meas = load_qasm3_circuit(path)
            qc_no_meas = strip_final_measurements(qc_meas)
            p_ideals.append(ideal_probabilities(qc_no_meas))
            circuits.append(_transpile_for_ionq(backend, qc_meas))
            circ_files.append(path.name)

        batch = 1 if str(cfg.get("backend", "")).startswith("qpu") else IONQ_BATCH_SIZE
        counts_list, job_ids = _submit_chunks(
            backend, circuits, shots=SHOTS_XEB, batch_size=batch,
        )
        new_job_ids.extend(job_ids)

        for k, cdict in enumerate(counts_list):
            stem = Path(circ_files[k]).stem
            (counts_dir / f"{stem}.json").write_text(json.dumps(cdict))

    dt = time.time() - t0
    job_ids_path = out_xeb / "job_ids.txt"
    prior_ids: List[str] = []
    if job_ids_path.is_file():
        prior_ids = [ln.strip() for ln in job_ids_path.read_text().splitlines() if ln.strip()]
    all_job_ids = prior_ids + new_job_ids
    job_ids_path.write_text("\n".join(all_job_ids) + ("\n" if all_job_ids else ""))

    summary = rebuild_xeb_summary_from_disk(cfg, out_xeb, qasm_dir, wall_time_s=dt)
    tag = _ionq_backend_name(backend)
    if cfg.get("noise_model"):
        tag += f" / noise={cfg['noise_model']}"
    print(f"  [XEB] {tag}: fit={summary['fit_A_f']}  time={dt:.1f}s  -> {out_xeb.resolve()}")
    return summary


def run_xeb_device(cfg: Dict[str, Any]) -> Path:
    slug = _device_slug(cfg)
    out = OUTPUT_ROOT / slug
    out_xeb = out / "xeb"
    qasm_dir = _resolve_qasm_dir()
    tag = cfg["backend"] + (f" + noise={cfg['noise_model']}" if cfg.get("noise_model") else "")
    print(f"\n=== XEB on {tag}  (slug={slug}) ===")
    print(f"     QASM: {qasm_dir}")
    run_xeb_ionq(cfg, out_xeb, qasm_dir)

    index_path = out / "device_index.json"
    index: Dict[str, Any] = {}
    if index_path.is_file():
        index = json.loads(index_path.read_text())
    index.update(
        {
            "backend_name_arg": cfg["backend"],
            "noise_model": cfg.get("noise_model"),
            "slug": slug,
            "qasm_dir": str(qasm_dir),
            "xeb_dir": str(out_xeb.resolve()),
        }
    )
    index_path.write_text(json.dumps(index, indent=2))
    print(f"=== finished -> {out.resolve()} ===\n")
    return out


def run_all_xeb_devices() -> None:
    if not IONQ_DEVICES:
        raise ValueError("IONQ_DEVICES is empty.")
    master: Dict[str, str] = {}
    if (OUTPUT_ROOT / "all_devices_index.json").is_file():
        master = json.loads((OUTPUT_ROOT / "all_devices_index.json").read_text())
    for cfg in IONQ_DEVICES:
        p = run_xeb_device(cfg)
        master[_device_slug(cfg)] = str(p.resolve())
    (OUTPUT_ROOT / "all_devices_index.json").write_text(json.dumps(master, indent=2))
    print("Master index:", (OUTPUT_ROOT / "all_devices_index.json").resolve())


## 1) Sanity-check XEB QASM

Confirms the QASM folder, lists depths, and spot-checks one circuit + ideal distribution.


In [ ]:
qasm_dir = _resolve_qasm_dir()
print("QASM_DIR =", qasm_dir)

groups = discover_xeb_files(qasm_dir)
depths = sorted(groups.keys())
print("depths:", depths)
for d in depths:
    print(f"  d={d:02d}: {len(groups[d])} circuit(s)")
    for p in groups[d][:3]:
        dd, kk = _parse_xeb_depth_idx(p)
        print(f"       {p.name} (d={dd}, k={kk})")
    if len(groups[d]) > 3:
        print(f"       ... ({len(groups[d]) - 3} more)")

sample = groups[depths[0]][0]
qc = load_qasm3_circuit(sample)
qc_nm = strip_final_measurements(qc)
p_ideal = ideal_probabilities(qc_nm)
print(
    f"\nSample {sample.name}: {qc.num_qubits}q, depth={qc.depth()}, size={qc.size()}; "
    f"|p_ideal| sum = {p_ideal.sum():.6f}"
)


## 2) Run XEB on all backends in `IONQ_DEVICES`

Simulator first (free). QPU jobs may queue for hours — job IDs are saved under
`<device>/xeb/job_ids.txt` for recovery.


In [ ]:
run_all_xeb_devices()


## 3) Optional — one backend only

Uncomment and run this cell instead of section 2.


In [ ]:
# run_xeb_device({"backend": "simulator", "slug": "simulator_ideal"})
# run_xeb_device({"backend": "qpu.forte-1", "slug": "qpu_forte_1"})


## 4) Recover finished QPU jobs → `counts/`

Run this **before** resuming the remaining circuits if counts were never saved locally.
Each QPU job is one circuit — map `(job_id, stem)` in submission order (`c000`, `c001`, …).
Counts are written to `<slug>/xeb/counts/` and `job_ids.txt` is updated.


In [ ]:
# Recover counts from finished QPU jobs (1 circuit per IonQ job).
RECOVER_BACKEND = "qpu.forte-1"
RECOVER_SLUG = "qpu_forte_1"
RECOVER_JOBS: List[Tuple[str, str]] = [
    ("019f45b5-9e98-77ac-be02-e26799dd00b0", "xeb_d01_c000"),
    ("019f45ba-d0e6-749d-9b95-b3256ff83a98", "xeb_d01_c001"),
    ("019f45c7-fbbb-7119-beff-494552eb8de1", "xeb_d01_c002"),
    ("019f45da-2b93-7402-bbbd-a45a4de024f0", "xeb_d01_c003"),
    ("019f45e2-d6ef-7768-bb6a-331f9cf018cb", "xeb_d01_c004"),
    ("019f45ea-efed-729d-8faf-9b3dc70cc46f", "xeb_d01_c005"),
]

out_xeb = OUTPUT_ROOT / RECOVER_SLUG / "xeb"
counts_dir = out_xeb / "counts"
counts_dir.mkdir(parents=True, exist_ok=True)
qasm_dir = _resolve_qasm_dir()
backend = _get_ionq_backend({"backend": RECOVER_BACKEND})

recovered_ids: List[str] = []
for job_id, stem in RECOVER_JOBS:
    out_path = counts_dir / f"{stem}.json"
    if out_path.is_file():
        print(f"[skip] {stem} (already on disk)")
        recovered_ids.append(job_id)
        continue

    qasm_path = qasm_dir / f"{stem}.qasm"
    if not qasm_path.is_file():
        raise FileNotFoundError(f"Missing QASM for recovery: {qasm_path}")

    qc_meas = load_qasm3_circuit(qasm_path)
    circ = _transpile_for_ionq(backend, qc_meas)
    job = backend.retrieve_job(job_id)
    print(f"{stem}: job_id={job_id}  status={job.status()}")
    job.wait_for_final_state(timeout=IONQ_JOB_TIMEOUT_S, wait=IONQ_POLL_S)
    counts_list = _fetch_chunk_counts_ionq(job, [circ], SHOTS_XEB)
    cdict = counts_list[0]
    out_path.write_text(json.dumps(cdict))
    print(f"  saved {out_path.name}  total_shots={sum(cdict.values())}")
    recovered_ids.append(job_id)

job_ids_path = out_xeb / "job_ids.txt"
prior_ids: List[str] = []
if job_ids_path.is_file():
    prior_ids = [ln.strip() for ln in job_ids_path.read_text().splitlines() if ln.strip()]
all_ids = prior_ids + [jid for jid in recovered_ids if jid not in prior_ids]
job_ids_path.write_text("\n".join(all_ids) + ("\n" if all_ids else ""))
print(f"\nUpdated {job_ids_path} ({len(all_ids)} job id(s))")

rebuild_xeb_summary_from_disk(
    {"backend": RECOVER_BACKEND, "slug": RECOVER_SLUG},
    out_xeb,
    qasm_dir,
)


## 5) Rebuild merged summary from disk

Run after recovery + resume if you want to refresh `summary.json` / plot without
re-submitting anything. Merges **all** `counts/*.json` (recovered + newly run).


In [ ]:
MERGE_CFG = {"backend": "qpu.forte-1", "slug": "qpu_forte_1"}
MERGE_OUT = OUTPUT_ROOT / MERGE_CFG["slug"] / "xeb"
rebuild_xeb_summary_from_disk(MERGE_CFG, MERGE_OUT, _resolve_qasm_dir())


## Output layout

```
data/raw/ionq_native/<RUN_STAMP>/
  all_devices_index.json
  <device_slug>/
    device_index.json          # merged with QAOA/VQC entries if same RUN_STAMP
    xeb/
      summary.json
      per_depth_mean_sem.json
      per_depth_per_circuit.json
      counts/xeb_d01_c000.json
      job_ids.txt
      xeb_plot.png
```

Ideal Born references (no cloud run): `analysis/xeb_ideal_reference_from_qasm.csv`.
